# 01 — Prepare official Kroger API locations

Download or load Kroger locations from Kroger's official Locations API, validate them, and write the portable state, ZIP, and brand summaries. Only official API records are accepted.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "scripts" / "build_all.py").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT / "scripts"))
ROOT

## Official API download

Create `.env` from `.env.example` and add your Kroger developer credentials. Set `RUN_OFFICIAL_DOWNLOAD = True` to refresh the data. The downloader uses OAuth2, respects Kroger's daily request limit, and never writes credentials to output files.

In [ ]:
import subprocess, sys

RUN_OFFICIAL_DOWNLOAD = False  # Keep False when notebook 00 already succeeded
print(f'[1/5] API refresh requested: {RUN_OFFICIAL_DOWNLOAD}')

if RUN_OFFICIAL_DOWNLOAD:
    print('[2/5] Starting official Kroger API download...')
    result = subprocess.run(
        [sys.executable, str(ROOT / 'scripts' / 'download_official_api.py'), '--chain', 'KROGER'],
        cwd=ROOT,
    )
    if result.returncode:
        raise RuntimeError('Official Kroger API download failed. Check your .env credentials.')
    print('[2/5] Official API download complete.')
else:
    print('[2/5] Download skipped; using the successful output from notebook 00.')

In [ ]:
import pandas as pd
from build_all import write_summaries

print('[3/5] Checking for the official API output...')
official_path = ROOT / 'data' / 'processed' / 'kroger_official_locations.csv'
if not official_path.exists():
    raise FileNotFoundError(
        'Official API export not found. Configure .env, set RUN_OFFICIAL_DOWNLOAD=True, and run the previous cell.'
    )
print('[3/5] Official API output found. Loading and validating records...')
locations = pd.read_csv(official_path, dtype={'zip_code': 'string'}).fillna({'website':'', 'phone':'', 'address':'', 'city':''})
assert locations['source'].eq('Kroger official Locations API').all(), 'Non-official records detected'
print(f'[4/5] Validation passed: {len(locations):,} official locations across {locations.state.nunique()} states.')
state_summary, zip_summary, brand_summary = write_summaries(locations)
print(f'[5/5] Complete: wrote {len(state_summary)} state rows, {len(zip_summary):,} ZIP rows, and {len(brand_summary)} brand rows.')
locations.head()

## Quality checks

Missing values are reported explicitly. A missing ZIP is retained in the location file but excluded from the ZIP summary.

In [ ]:
locations[['location_id','brand','state','zip_code','status','status_basis','latitude','longitude']].isna().sum().to_frame('missing')

## Store status

`active` means Kroger's official Locations API returned the store on the collection date in `api_collected_at_utc`. The public API does not provide closed-store history or independent real-time operating confirmation.

In [ ]:
locations.groupby(['status', 'status_basis'], dropna=False).size().to_frame('locations')

In [ ]:
locations.loc[locations['zip_code'].eq(''), ['name','brand','city','state','website']].head(20)

In [ ]:
assert locations['state'].str.fullmatch(r'[A-Z]{2}').all()
assert locations['latitude'].between(18, 72).all()
assert locations['longitude'].between(-180, -60).all()
print('Basic U.S. coordinate and state checks passed.')